In [2]:
import pandas as pd

import numpy as np
import re

In [3]:

# 📂 File gốc
csv_path = r"D:\Recommend_System\data\recipes\merged_recipes.csv"
output_path = csv_path.replace(".csv", "_clean.csv")

# 1️⃣ Đọc file gốc an toàn
df = pd.read_csv(csv_path)

print(f"📂 Đã đọc file: {csv_path}")
print(f"📊 Tổng số dòng ban đầu: {len(df)} | Số cột: {len(df.columns)}")

# 2️⃣ Làm sạch dữ liệu
required_cols = ["title", "description", "url", "ingredients", "instructions", "servings"]
required_cols = [c for c in required_cols if c in df.columns]

df_clean = df.dropna(subset=required_cols)
df_clean = df_clean[
    (df_clean["title"].astype(str).str.strip() != "") &
    (df_clean["url"].astype(str).str.strip() != "")
]

print(f"🧹 Đã xoá {len(df) - len(df_clean)} dòng thiếu dữ liệu quan trọng.")
print(f"✅ Còn lại {len(df_clean)} dòng hợp lệ sau khi lọc.")

# 3️⃣ Xoá trùng title
before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title"], keep="first")
print(f"🧩 Đã xoá {before_dup - len(df_clean)} dòng trùng title.")
print(f"📈 Còn lại {len(df_clean)} dòng sau khi xoá trùng.")

# 4️⃣ Xoá cột 'source_file' nếu có
if "source_file" in df_clean.columns:
    df_clean = df_clean.drop(columns=["source_file"])
    print("🗑️ Đã xoá cột 'source_file'.")

# 5️⃣ Sắp xếp lại thứ tự cột
new_order = [
    "title", "url", "description",
    "prep_time", "cook_time", "total_time",
    "rating_value", "rating_count", "review_count"
]
remaining_cols = [c for c in df_clean.columns if c not in new_order]
df_clean = df_clean[new_order + remaining_cols]

print("📑 Thứ tự 10 cột đầu tiên:", df_clean.columns[:10].tolist())

# 6️⃣ Lưu file kết quả
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"🎉 File đã clean và lưu tại: {output_path}")


📂 Đã đọc file: D:\Recommend_System\data\recipes\merged_recipes.csv
📊 Tổng số dòng ban đầu: 39776 | Số cột: 24
🧹 Đã xoá 34 dòng thiếu dữ liệu quan trọng.
✅ Còn lại 39742 dòng hợp lệ sau khi lọc.
🧩 Đã xoá 0 dòng trùng title.
📈 Còn lại 39742 dòng sau khi xoá trùng.
🗑️ Đã xoá cột 'source_file'.
📑 Thứ tự 10 cột đầu tiên: ['title', 'url', 'description', 'prep_time', 'cook_time', 'total_time', 'rating_value', 'rating_count', 'review_count', 'servings']
🎉 File đã clean và lưu tại: D:\Recommend_System\data\recipes\merged_recipes_clean.csv


In [4]:
# 📂 Đường dẫn tới file đã merge
csv_path = r"D:\Recommend_System\data\recipes\merged_recipes_clean.csv"  

# Đọc file
df = pd.read_csv(csv_path)

# Hiển thị thông tin cơ bản
print("✅ Đã đọc dữ liệu thành công!")
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
display(df.head(3))

✅ Đã đọc dữ liệu thành công!
Số dòng: 39742
Số cột: 23


,title,url,description,prep_time,cook_time,total_time,rating_value,rating_count,review_count,servings,...,ingredients,instructions,calories,protein,fat,carbohydrate,fiber,sugar,sodium,image_url
0,15-Minute Butter Gnocchi with Spicy Chili Cris...,https://www.allrecipes.com/15-minute-butter-gn...,"This gnocchi with chili crisp, capers, and Par...",5m,10m,15m,4.0,2.0,2.0,4 to 6,...,"[Ingredients] 2 pounds gnocchi , 6 tablespoons...",[Instructions] 1) Bring a large pan of water t...,507 kcal,14 g,19 g,69 g,4 g,1 g,385 mg,https://www.allrecipes.com/thmb/rtwG82TF-audw4...
1,15-Minute Chili Crisp Noodles,https://www.allrecipes.com/15-minute-chili-cri...,These 15-minute chili crisp noodles are spicy ...,10m,5m,15m,NaN,NaN,0.0,3,...,"[Ingredients] 5 cups water , 2 (3 ounce) packa...",[Instructions] 1) Fill a pot with water and br...,469 kcal,13 g,27 g,47 g,4 g,4 g,1758 mg,https://www.allrecipes.com/thmb/ZEzxRO4PfkqUoA...
2,15-Minute Creamy Garlic Basil Pasta,https://www.allrecipes.com/15-minute-creamy-ga...,This 15-minute creamy garlic basil pasta sauce...,3m,12m,15m,4.1,7.0,7.0,4,...,"[Ingredients] 8 ounces pasta, any type , 1 tab...",[Instructions] 1) Fill a large pot with lightl...,244 kcal,8 g,12 g,26 g,1 g,7 g,359 mg,https://www.allrecipes.com/thmb/QZ4ZKh7Qp22Hla...


In [5]:
# ===============================================

# 1️⃣ Đọc file
src = r"D:\Recommend_System\data\recipes\merged_recipes_clean.csv"
df = pd.read_csv(src, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {src} ({len(df)} dòng)")

if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong CSV!")

# 2️⃣ Danh sách cần gán 'World'
drop_list = {
    "Dessert", "Baking", "Dog Food", "Universal", "Authentic", 
    "Copycat", "Inspired", "World"
}

# 3️⃣ Hàm chuẩn hoá text
def normalize_text(s):
    if pd.isna(s):
        return ""
    return re.sub(r"\s+", " ", str(s).strip().lower())

# 4️⃣ Từ điển nhóm chuẩn (được gom nhóm theo danh sách 117 cuisine của bạn)
groups = {
    "Vietnamese": ["vietnamese"],
    "Chinese": ["chinese", "sichuan"],
    "Japanese": ["japanese"],
    "Korean": ["korean"],
    "Thai": ["thai"],
    "Indian": ["indian", "bangladeshi", "pakistani", "sri lankan"],
    "Italian": ["italian", "sicilian"],
    "French": ["french", "french canadian"],
    "British": ["british", "english", "uk", "scottish", "irish", "welsh"],
    "American": [
        "american", "u.s.", "north american", "native american", "southwestern",
        "southern", "hawaiian", "new england", "pennsylvania dutch", "amish",
        "tex mex", "tex-mex", "cajun", "creole"
    ],
    "Mexican": ["mexican", "latin", "latin american", "south american", "argentine", "brazilian", "venezuelan"],
    "Mediterranean": [
        "greek", "lebanese", "turkish", "israeli", "persian", "moroccan", 
        "egyptian", "tunisian", "middle eastern", "north african"
    ],
    "European": [
        "european", "austrian", "german", "swiss", "hungarian", "polish", "spanish", 
        "portuguese", "dutch", "danish", "norwegian", "finnish", "swedish", 
        "scandinavian", "russian", "ukrainian", "eastern european"
    ],
    "Australian": ["australian", "new zealand", "oceanic"],
    "Asian": [
        "asian", "east and southeast asian", "south and central asian",
        "asian inspired", "asian fusion", "filipino", "malaysian", "indonesian", 
        "singaporean"
    ],
    "African": ["african", "ethiopian", "west african", "south african", "east african"],
    "Caribbean": ["caribbean", "jamaican", "cuban", "puerto rican", "trinidad", "salvadoran", "colombian", "chilean", "peruvian"],
    "Jewish": ["jewish", "kosher"],
    "Fusion": ["fusion", "modern", "inspired"]
}

# 5️⃣ Chuẩn hoá từng recipe, giữ nguyên multi-cuisine
def normalize_cuisine_field(cuisine_field):
    if pd.isna(cuisine_field) or str(cuisine_field).strip() == "":
        return "World"

    cuisines = [c.strip() for c in str(cuisine_field).split(",") if c.strip()]
    normalized = []

    for c in cuisines:
        cname = normalize_text(c)
        # Nếu nằm trong drop_list hoặc dạng "Inspired" → World
        if any(dl.lower() == cname for dl in drop_list):
            normalized.append("World")
            continue

        matched = None
        for group, kws in groups.items():
            for kw in kws:
                if kw in cname:
                    matched = group
                    break
            if matched:
                break
        normalized.append(matched or "World")

    # Loại trùng, giữ thứ tự
    seen, result = set(), []
    for item in normalized:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return ", ".join(result)

# 6️⃣ Áp dụng
df["recipe_cuisine"] = df["recipe_cuisine"].apply(normalize_cuisine_field)

# 7️⃣ Lưu file mới
out_path = src.replace(".csv", "_normalized.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Đã chuẩn hoá cột 'recipe_cuisine' (fallback = 'World')")
print(f"💾 File lưu tại: {out_path}")

# 8️⃣ Thống kê tần suất
cuisine_counts = {}
for cell in df["recipe_cuisine"]:
    for c in [x.strip() for x in str(cell).split(",") if x.strip()]:
        cuisine_counts[c] = cuisine_counts.get(c, 0) + 1

print("\n📊 Top 20 cuisine phổ biến nhất:")
for c, cnt in sorted(cuisine_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{c:15} → {cnt:5}")


✅ Đã đọc file: D:\Recommend_System\data\recipes\merged_recipes_clean.csv (39742 dòng)

✅ Đã chuẩn hoá cột 'recipe_cuisine' (fallback = 'World')
💾 File lưu tại: D:\Recommend_System\data\recipes\merged_recipes_clean_normalized.csv

📊 Top 20 cuisine phổ biến nhất:
American        → 20051
World           → 10855
Italian         →  2234
Mexican         →  2031
Asian           →  1151
European        →  1051
Mediterranean   →   857
French          →   565
British         →   546
Indian          →   528
Caribbean       →   344
Chinese         →   296
Thai            →   223
Japanese        →   208
Fusion          →   170
Korean          →   132
Jewish          →   129
African         →   123
Vietnamese      →    70
Australian      →    61


In [6]:

# 1️⃣ Đọc file gốc
input_path = r"D:\Recommend_System\data\recipes\merged_recipes_clean_normalized.csv"
output_path = input_path.replace(".csv", "_time.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path}")
print(f"📦 Số dòng: {len(df)} | Số cột: {len(df.columns)}")
print(f"🔍 Số lượng giá trị thiếu:\n{df[["prep_time", "cook_time", "total_time"]].isnull().sum()}")
# 2️⃣ Chuyển về phút
def to_minutes(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = re.sub(r"[^0-9hms ]", "", s)
    if "h" in s:
        h = re.search(r"(\d+)\s*h", s)
        m = re.search(r"(\d+)\s*m", s)
        hours = int(h.group(1)) if h else 0
        mins = int(m.group(1)) if m else 0
        return hours * 60 + mins
    m = re.search(r"(\d+)", s)
    return int(m.group(1)) if m else np.nan

for c in ["prep_time", "cook_time", "total_time"]:
    df[c] = df[c].apply(to_minutes)

# 3️⃣ Logic đơn giản
# Nếu có total, gắn cho cook/prep nếu bị NaN
df.loc[df["total_time"].notna() & df["prep_time"].isna(), "prep_time"] = df["total_time"]
df.loc[df["total_time"].notna() & df["cook_time"].isna(), "cook_time"] = df["total_time"]

# Nếu total bị NaN → lấy cook hoặc prep
df.loc[df["total_time"].isna() & df["cook_time"].notna(), "total_time"] = df["cook_time"]
df.loc[df["total_time"].isna() & df["prep_time"].notna(), "total_time"] = df["prep_time"]

# 4️⃣ Format lại dạng 'Xm'
def fmt_m(x):
    return f"{int(x)}m" if pd.notna(x) else ""

for c in ["prep_time", "cook_time", "total_time"]:
    df[c] = df[c].apply(fmt_m)

# 5️⃣ Xuất file kết quả
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã xử lý thời gian và lưu tại:\n👉 {output_path}")
print("\n🔎 5 dòng đầu sau xử lý:")
print(df[["title", "prep_time", "cook_time", "total_time"]].head().to_string(index=False))


✅ Đã đọc file: D:\Recommend_System\data\recipes\merged_recipes_clean_normalized.csv
📦 Số dòng: 39742 | Số cột: 23
🔍 Số lượng giá trị thiếu:
prep_time     1398
cook_time     6746
total_time       0
dtype: int64
✅ Đã xử lý thời gian và lưu tại:
👉 D:\Recommend_System\data\recipes\merged_recipes_clean_normalized_time.csv

🔎 5 dòng đầu sau xử lý:
                                                               title prep_time cook_time total_time
15-Minute Butter Gnocchi with Spicy Chili Crisp, Capers and Parmesan        5m       10m        15m
                                       15-Minute Chili Crisp Noodles       10m        5m        15m
                                 15-Minute Creamy Garlic Basil Pasta        3m       12m        15m
              3-Ingredient Air Fryer Everything Bagel Chicken Strips        5m       15m        20m
                                       3-Ingredient Baked Pork Chops       10m       20m        30m


In [7]:
# ======================================================
# 🔧 Chuẩn hoá đơn vị nguyên liệu trong cột [ingredients]
# - ounce / oz  ➜  chuyển sang gram và xoá chữ ounce
# - tbsp / tsp  ➜  thêm số gram trong ngoặc (..g)
# - "1 20-ounce package ..." ➜ "1 (...g) package ..."
# ======================================================

import pandas as pd
import numpy as np
import re

# 1️⃣ Đọc file gốc
input_path = r"D:\Recommend_System\data\recipes\full_data.csv"
output_path = input_path.replace(".csv", "_ing.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path}")
print(f"📦 Số dòng: {len(df)} | Số cột: {len(df.columns)}")

# Kiểm tra cột ingredients
if "ingredients" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'ingredients' trong file!")

print(f"🔍 Ví dụ 3 dòng đầu:")
print(df["ingredients"].head(3).to_string(index=False))

# 2️⃣ Bảng quy đổi
conversion_table = {
    "ounce": 28.35, "ounces": 28.35, "oz": 28.35,
    "tbsp": 15, "tablespoon": 15, "tablespoons": 15,
    "tsp": 5, "teaspoon": 5, "teaspoons": 5,
}

# 3️⃣ Hàm chuẩn hoá 1 dòng
def normalize_ingredient_text(text):
    if pd.isna(text):
        return text
    s = str(text)

    # ➤ 0️⃣ Dạng phạm vi "(6- to 8-ounce)" → "(170–227g)"
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?)\s*[-–]?\s*(?:to)?\s*(\d+(?:\.\d+)?)\s*[- ]*(?:ounce|ounces|oz)\s*\)",
        lambda m: f"({round(float(m.group(1)) * 28.35)}–{round(float(m.group(2)) * 28.35)}g)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 1️⃣ "(12 fluid ounce)" → "(354.8ml)"
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(fluid\s*ounces?|fl\.?\s*oz)\s*\)",
        lambda m: f"({round(eval(m.group(1).replace(' ', '+')) * 29.57, 1)}ml)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 2️⃣ "8 fluid ounces" → "236.6ml"
    s = re.sub(
        r"\b(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(fluid\s*ounces?|fl\.?\s*oz)\b",
        lambda m: f"{round(eval(m.group(1).replace(' ', '+')) * 29.57, 1)}ml",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 3️⃣ "1 20-ounce package ..." → "1 (567g) package ..."
    s = re.sub(
        r"(\b\d+(?:\.\d+)?\b)\s+(\d+(?:\.\d+)?)[- ]?(ounce|oz)\b",
        lambda m: f"{m.group(1)} ({round(float(m.group(2)) * 28.35)}g)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 4️⃣ "(3-ounce)" hoặc "3-ounce" → "(85.1g)" hoặc "85.1 g"
    # có thể có ngoặc hoặc không
    s = re.sub(
        r"\(?(\d+(?:\.\d+)?)[- ]?(?:ounce|ounces|oz)\)?",
        lambda m: f"{round(float(m.group(1)) * 28.35, 1)} g",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 5️⃣ "4 tbsp" → "4tbsp (15g)"
    def add_g_parentheses(m):
        qty = m.group(1)
        unit = m.group(2).lower()
        grams = conversion_table[unit]
        return f"{qty}{unit} ({grams}g)"

    s = re.sub(
        r"\b(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(tbsp|tablespoon|tablespoons|tsp|teaspoon|teaspoons)\b",
        add_g_parentheses,
        s,
        flags=re.IGNORECASE
    )

    return s




# 4️⃣ Áp dụng vào toàn bộ cột ingredients
df["ingredients"] = df["ingredients"].apply(normalize_ingredient_text)

# 5️⃣ Xuất file kết quả
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã chuẩn hoá đơn vị trong ingredients và lưu tại:\n👉 {output_path}")
print("\n🔎 5 dòng đầu sau xử lý:")
print(df["ingredients"].head().to_string(index=False))


✅ Đã đọc file: D:\Recommend_System\data\recipes\full_data.csv
📦 Số dòng: 39742 | Số cột: 23
🔍 Ví dụ 3 dòng đầu:
[Ingredients] 2 pounds gnocchi , 6 tablespoons ...
[Ingredients] 5 cups water , 2 (3 ounce) packag...
[Ingredients] 8 ounces pasta, any type , 1 tabl...
✅ Đã chuẩn hoá đơn vị trong ingredients và lưu tại:
👉 D:\Recommend_System\data\recipes\full_data_ing.csv

🔎 5 dòng đầu sau xử lý:
[Ingredients] 2 pounds gnocchi , 6tablespoons (...
[Ingredients] 5 cups water , 2 85.1 g packages ...
[Ingredients] 226.8 gs pasta, any type , 1table...
[Ingredients] 1 1/4 pound fresh chicken tenders...
[Ingredients] 2 large eggs , 4 (170–227g) bone-...


In [8]:
# ======================================================
# 🔧 Chuẩn hoá đơn vị đo trong cột [ingredients] và [instructions]
# - ounce / oz  ➜  chuyển sang gram
# - fluid ounce / fl oz ➜ chuyển sang ml
# - tbsp / tsp  ➜ thêm số gram trong ngoặc (..g)
# ======================================================

import pandas as pd
import numpy as np
import re

# 1️⃣ Đọc file gốc
input_path = r"D:\Recommend_System\data\recipes\full_data.csv"
output_path = input_path.replace(".csv", "_ing.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path}")
print(f"📦 Số dòng: {len(df)} | Số cột: {len(df.columns)}")

# 2️⃣ Bảng quy đổi
conversion_table = {
    "ounce": 28.35, "ounces": 28.35, "oz": 28.35,
    "tbsp": 15, "tablespoon": 15, "tablespoons": 15,
    "tsp": 5, "teaspoon": 5, "teaspoons": 5,
}

# 3️⃣ Hàm chuẩn hoá đơn vị trong 1 đoạn text (áp dụng cho ingredients và instructions)
def normalize_units(text):
    if pd.isna(text):
        return text
    s = str(text)

    # ➤ (6- to 8-ounce) → (170–227g)
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?)\s*[-–]?\s*(?:to)?\s*(\d+(?:\.\d+)?)\s*[- ]*(?:ounce|ounces|oz)\s*\)",
        lambda m: f"({round(float(m.group(1))*28.35)}–{round(float(m.group(2))*28.35)}g)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ (12 fluid ounce) → (354.8ml)
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(fluid\s*ounces?|fl\.?\s*oz)\s*\)",
        lambda m: f"({round(eval(m.group(1).replace(' ', '+'))*29.57,1)}ml)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 8 fluid ounces → 236.6ml
    s = re.sub(
        r"\b(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(fluid\s*ounces?|fl\.?\s*oz)\b",
        lambda m: f"{round(eval(m.group(1).replace(' ', '+'))*29.57,1)}ml",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 1 20-ounce → 1 (567g)
    s = re.sub(
        r"(\b\d+(?:\.\d+)?\b)\s+(\d+(?:\.\d+)?)[- ]?(ounce|oz)\b",
        lambda m: f"{m.group(1)} ({round(float(m.group(2))*28.35)}g)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ (3-ounce) hoặc 3-ounce hoặc 3 ounce → 85.1 g
    s = re.sub(
        r"\(?(\d+(?:\.\d+)?)[- ]?(?:ounce|ounces|oz)\)?",
        lambda m: f"{round(float(m.group(1))*28.35,1)} g",
        s,
        flags=re.IGNORECASE
    )

    # ➤ tbsp / tsp → thêm (..g)
    def add_g_parentheses(m):
        qty = m.group(1)
        unit = m.group(2).lower()
        grams = conversion_table[unit]
        return f"{qty}{unit} ({grams}g)"

    s = re.sub(
        r"\b(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(tbsp|tablespoon|tablespoons|tsp|teaspoon|teaspoons)\b",
        add_g_parentheses,
        s,
        flags=re.IGNORECASE
    )

    return s


# 4️⃣ Áp dụng vào các cột nếu tồn tại
if "ingredients" in df.columns:
    df["ingredients"] = df["ingredients"].apply(normalize_units)
    print("✅ Đã chuẩn hoá đơn vị trong [ingredients]")

if "instructions" in df.columns or "introductions" in df.columns:
    col = "instructions" if "instructions" in df.columns else "introductions"
    df[col] = df[col].apply(normalize_units)
    print(f"✅ Đã chuẩn hoá đơn vị trong [{col}]")

# 5️⃣ Lưu file
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã xử lý xong và lưu tại:\n👉 {output_path}")


✅ Đã đọc file: D:\Recommend_System\data\recipes\full_data.csv
📦 Số dòng: 39742 | Số cột: 23
✅ Đã chuẩn hoá đơn vị trong [ingredients]
✅ Đã chuẩn hoá đơn vị trong [instructions]
✅ Đã xử lý xong và lưu tại:
👉 D:\Recommend_System\data\recipes\full_data_ing.csv


In [15]:
import pandas as pd

# 📂 Đường dẫn file
input_path = r"D:\Recommend_System\data\recipes\merged_recipes_1.csv"
output_path = input_path.replace(".csv", ".csv")

# 1️⃣ Đọc file
df = pd.read_csv(input_path, encoding="utf-8-sig")

# 2️⃣ Kiểm tra cột có chứa cuisine
if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong file!")

# 3️⃣ Thay thế giá trị
df["recipe_cuisine"] = df["recipe_cuisine"].replace("All", "World")

# (Tuỳ chọn) Nếu có ô NaN thì cũng gán thành "World"
df["recipe_cuisine"] = df["recipe_cuisine"].fillna("World")

# 4️⃣ Ghi lại file mới
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã chuyển toàn bộ 'All' → 'World' và lưu tại:\n👉 {output_path}")

# 5️⃣ In kiểm tra nhanh
print("\n🔎 5 dòng đầu sau khi đổi:")
print(df["recipe_cuisine"].head())


✅ Đã chuyển toàn bộ 'All' → 'World' và lưu tại:
👉 D:\Recommend_System\data\recipes\merged_recipes_1.csv

🔎 5 dòng đầu sau khi đổi:
0      Fusion
1       Asian
2    American
3    American
4    American
Name: recipe_cuisine, dtype: object


In [2]:
import pandas as pd
df = pd.read_csv("D:\\Recommend_System\\data\\recipes\\full_data_ing.csv", encoding="utf-8-sig")
print(len(df))

39742


In [7]:
import pandas as pd

path = r"D:\Recommend_System\data\users\users-survey.csv"  # hoặc đường dẫn của bạn
df = pd.read_csv(path)

# 🔍 In toàn bộ tên cột thật (có thể copy ra để dùng chính xác)
for i, c in enumerate(df.columns, 1):
    print(f"{i}. {repr(c)}")


1. 'Dấu thời gian'
2. 'Giới tính (Gender) '
3. '  Độ tuổi (Age)  '
4. '  Bạn có hạn chế chế độ ăn nào không? (Do you have any dietary restrictions?)  '
5. '  Bạn có dị ứng với thực phẩm nào không? (Do you have any food allergies?)  '
6. 'Bạn thường quan tâm đến loại bữa ăn nào? (Which type of meal do you usually care about?)'
7. 'Mức độ kỹ năng nấu ăn của bạn là gì? (Your cooking skill level)'
8. 'Bạn có bao nhiêu thời gian để nấu ăn trung bình? (How much time do you usually have to cook?)'
9. ' Bạn yêu thích món ăn của quốc gia nào? (Which country’s cuisine do you like?)'
10. 'Bạn thích những món ăn Việt Nam nào sau đây? (Which Vietnamese dishes do you like?)'
11. 'Bạn thích những món ăn Thái Lan nào sau đây? (Which Thai dishes do you like?)'
12. 'Bạn thích những món ăn Trung Quốc nào sau đây? (Which Chinese dishes do you like?)  '
13. 'Bạn thích những món ăn Pháp nào sau đây? (Which French dishes do you like?)'
14. 'Bạn thích những món ăn Ấn Độ nào sau đây? (Which Indian dishes do yo

In [8]:
# 📘 Bước 1: Import thư viện
import pandas as pd

# 📘 Bước 2: Đọc file CSV
path = r"D:\Recommend_System\data\users\users-survey.csv"  # chỉnh lại nếu file ở nơi khác
df = pd.read_csv(path)

# 📘 Bước 3: Kiểm tra tên cột chính xác
df.columns.tolist()
# 📘 Bước 4: Liệt kê toàn bộ nội dung trong cột
col_name = "Có món ăn nào bạn thích mà chưa được đề cập trong danh sách trên không? (nếu có)\nIs there any dish you like that was not mentioned in the list above?"
unique_answers = df[col_name].dropna().unique()

# 📘 Bước 5: Hiển thị kết quả
for i, val in enumerate(unique_answers, 1):
    print(f"{i}. {val}")



1. không
2. Có nhiều, liệt kê ko hết
3. Bún đậu mắm tôm
4. Panna cotta
5. Bì Bún (Việt Nam)
6. Cao lầu, bún bò Huế
7. No
8. Nhiều lắm
9. Bánh tráng trộn
10. Hong
11. Không
12. Ko có
13. Cơm sườn 
14. Bún đậu mắm tôm và takoyaky
15. Mì cayyy
16. Cơm Cuộn Hàn Quốc
17. Hog
18. Bún Bò
19. Không có
20. Gỏi cuốn
21. bánh tacos Pháp
22. Takoyaki
23. Tất cả các món Việt Nam
24. Lobster
25. không, món nào cũng thích 
26. Bột Masala của Ấn Độ
27. Nấm kim châm , nấm đùi gà và mì cay
28. Canh chua
29. bún bò
30. Cá viên chiên 
31. Coem Tấm
32. tôi thích tất cả các món ăn trên thế giới💗🥰
33. Mì cay
34. Masala là ngon nhất thế giới
35. Hamburger và phô mai 
36. không có
37. bánh khọt,bánh đúc nóng,quảng,súp don,ram bắp,cơm chiên dương châu,bột chiên
38. Bún bò, sườn xào chua ngọt, cơm sườn, joliebee 
39. cà ri ấn độ
40. Cơm cuộn tam giác (Hàn), Natto(Nhật),tacos(pháp),burito(mexico),Bún đậu( việt nam), sandwich (mỹ)
41. Bánh canh
42. mohammad salah, asalamulakum cà ra tu ni
43. ko
44. Cơm chiên dươn

In [9]:
import pandas as pd

# 📁 Đường dẫn input và output
file1 = r"D:\Recommend_System\data\recipes\full_data_ing.csv"
file2 = r"D:\Recommend_System\data\recipes\combined_hungryhuy_vickypham_simulated_ratings_vn.csv"
output_path = r"D:\Recommend_System\data\recipes\all_recipes_combined.csv"

# 📘 Đọc dữ liệu
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

print("🔹 File 1:", df1.shape)
print("🔹 File 2:", df2.shape)

# 🧩 Thêm cột còn thiếu để đồng bộ
for col in df2.columns:
    if col not in df1.columns:
        print(f"🧩 Thêm cột {col} vào file1")
        df1[col] = None

for col in df1.columns:
    if col not in df2.columns:
        print(f"🧩 Thêm cột {col} vào file2")
        df2[col] = None

# 🔄 Sắp xếp lại cùng thứ tự cột
df2 = df2[df1.columns]

# 🧷 Gộp dữ liệu
combined = pd.concat([df1, df2], ignore_index=True)

# 🚫 Xoá trùng (nếu cùng title và url)
if "title" in combined.columns and "url" in combined.columns:
    before = combined.shape[0]
    combined.drop_duplicates(subset=["title", "url"], inplace=True)
    after = combined.shape[0]
    print(f"✅ Đã xoá {before - after} dòng trùng lặp")

# 💾 Lưu file hợp nhất
combined.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"🎉 File đã được gộp và lưu tại: {output_path}")
print("📦 Kích thước cuối cùng:", combined.shape)


🔹 File 1: (39742, 23)
🔹 File 2: (457, 24)
🧩 Thêm cột is_vn_dish vào file1
✅ Đã xoá 0 dòng trùng lặp
🎉 File đã được gộp và lưu tại: D:\Recommend_System\data\recipes\all_recipes_combined.csv
📦 Kích thước cuối cùng: (40199, 24)
